# HW2 Part 2.2: Storing OLTP Events in a Data Lake

Create a new S3 bucket programmatically (via **Boto3**, the Python interface to the AWS API), then load the hourly `.parquet` files from the local `data` subfolder into a `wikipedia-hourly` "subfolder" within it.

This is the **Data Lake** pattern: raw, unstructured (OLTP) event data is tossed into a large container, to be fished out and transformed into OLAP data later.

In [1]:
import os
import boto3
from botocore.exceptions import ClientError

NETID = "me878"
BUCKET = f"dsan6000-{NETID}"      # -> s3://dsan6000-me878
DEST_PREFIX = "wikipedia-hourly"
DATA_DIR = "data"

s3 = boto3.client("s3")
print(f"target bucket: s3://{BUCKET}")

target bucket: s3://dsan6000-me878


## Create the bucket

No `LocationConstraint` is passed, which is correct for `us-east-1` (passing one there is an error; every other region requires it).

The `try`/`except` makes this cell safe to re-run: a second run reports that the bucket already exists rather than raising.

In [2]:
try:
    s3.create_bucket(Bucket=BUCKET)
    print(f"Created s3://{BUCKET}")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket already exists: {code}")
    else:
        raise

Created s3://dsan6000-me878


## Upload the hourly parquet files into the `wikipedia-hourly` subfolder

In [3]:
files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".parquet"))
print(f"uploading {len(files)} files...")

for filename in files:
    key = f"{DEST_PREFIX}/{filename}"
    s3.upload_file(os.path.join(DATA_DIR, filename), BUCKET, key)
    print(f"uploaded {key}")

uploading 24 files...
uploaded wikipedia-hourly/20260901_040000.parquet
uploaded wikipedia-hourly/20260901_050000.parquet
uploaded wikipedia-hourly/20260901_060000.parquet


uploaded wikipedia-hourly/20260901_070000.parquet
uploaded wikipedia-hourly/20260901_080000.parquet
uploaded wikipedia-hourly/20260901_090000.parquet
uploaded wikipedia-hourly/20260901_100000.parquet


uploaded wikipedia-hourly/20260901_110000.parquet
uploaded wikipedia-hourly/20260901_120000.parquet
uploaded wikipedia-hourly/20260901_130000.parquet
uploaded wikipedia-hourly/20260901_140000.parquet


uploaded wikipedia-hourly/20260901_150000.parquet
uploaded wikipedia-hourly/20260901_160000.parquet
uploaded wikipedia-hourly/20260901_170000.parquet
uploaded wikipedia-hourly/20260901_180000.parquet


uploaded wikipedia-hourly/20260901_190000.parquet
uploaded wikipedia-hourly/20260901_200000.parquet
uploaded wikipedia-hourly/20260901_210000.parquet
uploaded wikipedia-hourly/20260901_220000.parquet


uploaded wikipedia-hourly/20260901_230000.parquet
uploaded wikipedia-hourly/20260902_000000.parquet
uploaded wikipedia-hourly/20260902_010000.parquet
uploaded wikipedia-hourly/20260902_020000.parquet


uploaded wikipedia-hourly/20260902_030000.parquet


## Verify

Confirm the objects landed, and confirm the bucket is **not** publicly accessible (the default for a bucket created via the API — no custom setting was applied).

In [4]:
paginator = s3.get_paginator("list_objects_v2")
uploaded = [
    o["Key"]
    for p in paginator.paginate(Bucket=BUCKET, Prefix=DEST_PREFIX)
    for o in p.get("Contents", [])
]
print(f"{len(uploaded)} objects under s3://{BUCKET}/{DEST_PREFIX}/")
print(uploaded[:3])
print("...")
print(uploaded[-3:])

24 objects under s3://dsan6000-me878/wikipedia-hourly/
['wikipedia-hourly/20260901_040000.parquet', 'wikipedia-hourly/20260901_050000.parquet', 'wikipedia-hourly/20260901_060000.parquet']
...
['wikipedia-hourly/20260902_010000.parquet', 'wikipedia-hourly/20260902_020000.parquet', 'wikipedia-hourly/20260902_030000.parquet']


In [5]:
acc = s3.get_public_access_block(Bucket=BUCKET)["PublicAccessBlockConfiguration"]
print("public access block settings:")
for k, v in acc.items():
    print(f"  {k}: {v}")

public access block settings:
  BlockPublicAcls: True
  IgnorePublicAcls: True
  BlockPublicPolicy: True
  RestrictPublicBuckets: True
